In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import subprocess

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/prod_traj/_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dmrs.shape

In [ ]:
dmrs.head()

In [ ]:
dms = 3
filt_dmrs = dmrs[dmrs['number_of_dms'] > dms].copy()
filt_dmrs.shape

In [ ]:
filt_dmrs['size'] = filt_dmrs['end'] - filt_dmrs['start']
filt_dmrs['size'].mean()

In [ ]:
sns.histplot(filt_dmrs['size'])

In [ ]:
filt_dmrs

In [ ]:
filt_dmrs['hypomethylated_samples'].value_counts()

In [ ]:
filt_dmrs['hypermethylated_samples'].fillna('', inplace=True)
filt_dmrs['hypomethylated_samples'].fillna('', inplace=True)
filt_dmrs['hypermethylated_samples'] = filt_dmrs['hypermethylated_samples'].str.split(',')
filt_dmrs['hypomethylated_samples'] = filt_dmrs['hypomethylated_samples'].str.split(',')
filt_dmrs = filt_dmrs.explode('hypermethylated_samples').reset_index(drop=True)
filt_dmrs = filt_dmrs.explode('hypomethylated_samples').reset_index(drop=True)
filt_dmrs.index = [f'{x}_{y}' for x,y in zip(filt_dmrs['#chr'], filt_dmrs['start'])]
filt_dmrs.shape

In [ ]:
filt_dmrs.head()

In [ ]:
filt_dmrs['hypomethylated_samples'].value_counts()

In [ ]:
outdir = f'{projdir}/csv/dmrs/rna_cluster/prod_traj'
filt_dmrs.to_csv(f'{outdir}/dmrs.header.bed', sep='\t')
filt_dmrs.to_csv(f'{outdir}/dmrs.bed', sep='\t', header=None, index=None)

In [ ]:
subprocess.run(f'bgzip -f {outdir}/dmrs.bed', shell=True)
subprocess.run(f'tabix -p bed {outdir}/dmrs.bed.gz', shell=True)
filt_dmrs.to_csv(f'{outdir}/dmrs.bed', sep='\t', header=None, index=None)

In [ ]:
prod_traj = ['Start', 'Sendai', 'Inter1', 'Inter2', 'IPS']

In [ ]:
outdir = f'{projdir}/csv/dmrs/rna_cluster/prod_traj/by_group/'

def write_filt_dmrs(s):
    hypo_filt_dmrs = filt_dmrs[filt_dmrs['hypomethylated_samples'] == s]
    hyper_filt_dmrs = filt_dmrs[filt_dmrs['hypermethylated_samples'] == s]
    hypo_filt_dmrs = hypo_filt_dmrs[~hypo_filt_dmrs.index.duplicated()]
    hyper_filt_dmrs = hyper_filt_dmrs[~hyper_filt_dmrs.index.duplicated()]
    hypo_filt_dmrs.to_csv(f'{outdir}/{s}_hypo.csv', sep='\t', header=None, index=None)
    hyper_filt_dmrs.to_csv(f'{outdir}/{s}_hyper.csv', sep='\t', header=None, index=None)
    

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(write_filt_dmrs, prod_traj), total=len(prod_traj)))

In [ ]:
!date